# C. elegans — Linking Notebook: fastC-GC*

Prerequisites: `graph_states`, `community`, `manifold`, and `cgc_star` `connectivity` stages for the selected run, plus the dataset `shared/recording` checkpoint.
This notebook preserves the anchor-aware embargo and activity-covariate controls in the resumed stage artifact.
This lane is directed, so the reported source/target readouts keep their signed directional semantics.
Progress: long stages print checkpoint-aware timing; iterative stages also display live progress bars. Reused, failed, running, and missing checkpoints are reported explicitly.

Environment: select the project Python 3.14 `.venv` kernel. Install the locked packages with `uv sync --python 3.14 --locked --extra dev --extra csl --extra community --extra notebooks` from the repository root. The setup cell checks installed package versions; no upstream repository clone is required. Stored outputs may predate the backend migration; rerun the notebook to produce results with the current backend.


In [1]:
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    _REPOSITORY_ROOT = next(
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "notebooks" / "_shared.py").is_file()
    )
except StopIteration as error:
    raise RuntimeError(
        "Could not locate the effectome repository from the notebook working directory"
    ) from error
sys.path[:0] = [
    path
    for path in (str(_REPOSITORY_ROOT / "src"), str(_REPOSITORY_ROOT))
    if path not in sys.path
]


if sys.version_info[:2] != (3, 14):
    raise RuntimeError(
        "Select the project Python 3.14 .venv kernel before running this notebook. "
        f"Current interpreter: {sys.executable}"
    )
_BACKEND_DISTRIBUTIONS = ("fastcgc", "tigramite", "scikit-learn", "kmedoids", "igraph", "leidenalg")
try:
    _BACKEND_VERSIONS = {name: version(name) for name in _BACKEND_DISTRIBUTIONS}
except PackageNotFoundError as error:
    raise RuntimeError(
        "The notebook kernel is missing a required installed backend. Run "
        "uv sync --python 3.14 --locked --extra dev --extra csl --extra community --extra notebooks "
        "from the repository root, then restart the project .venv kernel."
    ) from error
print("Notebook interpreter:", sys.executable)
print("Installed backends:", _BACKEND_VERSIONS)


Notebook interpreter: /Users/sadiqadedayo/Documents/projects/effectome-to-behavior/effectome-to-behavior/.venv/bin/python
Installed backends: {'fastcgc': '0.1.0', 'tigramite': '5.2.10.1', 'scikit-learn': '1.9.0', 'kmedoids': '0.5.5', 'igraph': '1.0.0', 'leidenalg': '0.12.0'}


In [2]:
from notebooks._shared import PROJECT_ROOT, make_run, print_notebook_header, print_stage_status, run_linking

DATASET_ID = "c_elegans"
DATASET_OUTPUT_ID = "bundle-net-c-elegans"
RECORDING_ID = "bundle-net-worm-0"
RUN_ID = "reference"
METHOD = "cgc_star"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "analysis" / DATASET_OUTPUT_ID / RECORDING_ID
RECORDING_PATH = OUTPUT_ROOT / RUN_ID / "shared" / "stages" / "recording" / "artifact.pkl"
FORCE = False

LINKING_CFG = {
    "behavior_keys": ["motif"],
    "n_null": 1000,
    "n_bins": 5,
    "max_lag": 10,
    "positive_lag": 1,
    "n_folds": 5,
    "embargo": 4,
    "group_by": "recording",
    "lag_extension": 0,
    "preprocessing_past_support": 0,
    "preprocessing_future_support": 0,
    "allow_global_preprocessing": False,
    "null_kind": "circular_shift",
    "block_length": 8,
    "seed": 42,
}

run = make_run(OUTPUT_ROOT, RUN_ID, METHOD)
print_notebook_header(
    run,
    title=f"{METHOD} linking",
    expected_stages=["recording_input", "linking"],
    force=FORCE,
)


/Users/sadiqadedayo/Documents/projects/effectome-to-behavior/effectome-to-behavior/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cgc_star linking
run_id=reference method=cgc_star
run_dir=/Users/sadiqadedayo/Documents/projects/effectome-to-behavior/effectome-to-behavior/outputs/analysis/bundle-net-c-elegans/bundle-net-worm-0/reference/cgc_star
resume mode: FORCE=False, matching completed checkpoints will be reused
Rerun interrupted notebooks from the top; completed stages resume from checkpoints.
Stage status for reference/cgc_star:
  - community: completed (signature=1ed371e192, artifact=yes)
  - connectivity: completed (signature=6918ffd92f, artifact=yes)
  - connectivity.chunk-00000: completed (signature=3f5ca58960, artifact=yes)
  - connectivity.chunk-00001: completed (signature=28cfd3a16b, artifact=yes)
  - connectivity.chunk-00002: completed (signature=af336ee3d3, artifact=yes)
  - connectivity.chunk-00003: completed (signature=4526b4361f, artifact=yes)
  - connectivity.chunk-00004: completed (signature=58a6fbce76, artifact=yes)
  - connectivity.chunk-00005: completed (signature=01e20d7818, artifact=yes)
  

In [3]:
report = run_linking(
    run,
    recording_path=RECORDING_PATH,
    linking_cfg=LINKING_CFG,
    force=FORCE,
)
if report.get("status") == "unavailable":
    print("Linking unavailable:", report["reason"])
else:
    print(len(report["decoding"]))
    print(report["splitter"])
    print(report["positive_lag_incremental"].keys())
    print(report["effectome_to_future_manifold"])
print_stage_status(run)


[reference/cgc_star] recording_input: checking checkpoint; computing if needed...
[reference/cgc_star] recording_input: reused checkpoint in 0.0s
[reference/cgc_star] linking: checking checkpoint; computing if needed...


Switching-manifold null: 100%|██████████| 1000/1000 [00:03<00:00, 267.28draw/s]


[linking] motif: decoding 5 feature sets with blocked cross-validation
[linking] motif: decoding connectivity
[linking] motif: decoding state
[linking] motif: decoding community
[linking] motif: decoding manifold_dynamics
[linking] motif: decoding activity_magnitude


motif association null: 100%|██████████| 1000/1000 [00:02<00:00, 434.23draw/s]


[reference/cgc_star] linking: completed in 20.4s
5
{'mode': 'anchor_aware_grouped_purged', 'group_by': 'recording', 'continuity_boundaries': 'recording_hard_gap_and_sparse_bad_frame', 'embargo': 4, 'raw_dependency_support': {'lag_extension': 0, 'preprocessing_past': 0, 'preprocessing_future': 0, 'preprocess_kind': 'finite'}}
dict_keys(['motif'])
{'lag': 1, 'n_transitions': 181, 'outcome': 'euclidean_norm_of_z_future_minus_z_current', 'raw_effectome_decode': DecodeResult(feature_set='connectivity', behavior_key='future_manifold_displacement', task='regression', mean_score=nan, std_score=nan, n_folds=0, embargo=4), 'coordinate_contract': 'magnitude_is_rotation_reflection_invariant; transitions never cross folds, recordings, or declared gaps', 'claim_boundary': 'prospective_prediction_not_interventional_causation', 'community_reconfiguration_increment': IncrementalDecodeResult(baseline_score=nan, full_score=nan, gain=nan, n_folds=0, embargo=4), 'community_feature_contract': 'switching_fra